# Análise Epidemiológica de Vírus Respiratórios e Gripe (InfoGripe / Fiocruz)

O **InfoGripe** é uma iniciativa desenvolvida pelo Programa de Computação Científica da **Fiocruz** (PROCC/Fiocruz) em parceria com a Escola de Matemática Aplicada da Fundação Getulio Vargas (**EMap/FGV**) e a Coordenação-Geral de Vigilância de Doenças e Agravos Transmissíveis (**CGVDI/MS**).

### Objetivos deste Estudo:
1. **Panorama Nacional:** Acompanhar as séries temporais dos principais agentes etiológicos de SRAG (Síndrome Respiratória Aguda Grave).
2. **Foco em Gripes (Influenza A e B):** Identificar a sazonalidade e os picos anuais das gripes.
3. **Comparativo Regional:** Analisar os estados com maior incidência de gripe.
4. **Avaliação do Nowcasting:** Compreender a diferença entre os casos já registrados (brutos) e os casos estimados pela Fiocruz (com correção de atraso de notificação).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações visuais
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('tab10')

## 1. Carregamento e Preparação dos Dados

Os dados podem ser lidos diretamente da pasta `dados/` (baixados via `download_dados.py`) ou alternativamente da URL oficial do repositório da Fiocruz.

In [ ]:
caminho_csv = os.path.join('dados', 'virus_brasil_estados.csv')

if not os.path.exists(caminho_csv):
    url_oficial = 'https://raw.githubusercontent.com/infogripe/Boletim_InfoGripe/main/Dados/InfoGripe/estados_e_pais_serie_estimativas_virus_sem_filtro_febre.csv'
    print('Arquivo local não encontrado. Carregando diretamente do InfoGripe/Fiocruz...')
    df_virus = pd.read_csv(url_oficial, sep=';')
else:
    df_virus = pd.read_csv(caminho_csv, sep=';')

# Padronização dos nomes das colunas
df_virus.columns = [
    'sg_uf_not', 'uf_sigla', 'ano_epi', 'semana_epi', 'virus', 
    'mediana', 'li', 'ls', 'casos_registrados', 'casos_estimados', 'media_movel'
]

# Mapeamento dos códigos dos vírus para nomes legíveis
mapa_virus = {
    'FLU_A': 'Influenza A (Gripe A)',
    'FLU_B': 'Influenza B (Gripe B)',
    'SARS2': 'COVID-19 (SARS-CoV-2)',
    'VSR': 'Vírus Sincicial Respiratório (VSR)',
    'RINO': 'Rinovírus',
    'METAP': 'Metapneumovírus'
}
df_virus['virus_nome'] = df_virus['virus'].map(mapa_virus).fillna(df_virus['virus'])

# Período formatado (Ano-Semana)
df_virus['ano_semana'] = df_virus['ano_epi'].astype(str) + '-SE' + df_virus['semana_epi'].astype(str).str.zfill(2)

print('Dimensões do dataset:', df_virus.shape)
print('Anos disponíveis:', sorted(df_virus['ano_epi'].unique()))
df_virus.head()

## 2. Panorama Nacional: Séries Temporais dos Vírus Respiratórios

Visualização da evolução semanal dos casos estimados em âmbito nacional (`uf_sigla == 'BR'`).

In [ ]:
df_br = df_virus[df_virus['uf_sigla'] == 'BR'].sort_values(['ano_epi', 'semana_epi'])

pivot_br = df_br.pivot_table(
    index='ano_semana', 
    columns='virus_nome', 
    values='casos_estimados', 
    aggfunc='sum'
).fillna(0)

plt.figure(figsize=(15, 6))
for col in pivot_br.columns:
    plt.plot(pivot_br.index, pivot_br[col], label=col, linewidth=2)

indices_ticks = np.linspace(0, len(pivot_br) - 1, 14, dtype=int)
plt.xticks(indices_ticks, pivot_br.index[indices_ticks], rotation=45)
plt.title('InfoGripe Fiocruz - Casos Estimados de SRAG por Vírus no Brasil', fontsize=14, fontweight='bold')
plt.xlabel('Ano e Semana Epidemiológica')
plt.ylabel('Casos Estimados')
plt.legend(title='Vírus', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Foco nas Gripes: Influenza A vs Influenza B

Comparamos especificamente a dinâmica da **Influenza A** contra a **Influenza B** ao longo das semanas epidemiológicas.

In [ ]:
df_gripe_br = df_br[df_br['virus'].isin(['FLU_A', 'FLU_B'])]

plt.figure(figsize=(15, 6))
cores_gripe = {'Influenza A (Gripe A)': '#d62728', 'Influenza B (Gripe B)': '#1f77b4'}
sns.lineplot(
    data=df_gripe_br, 
    x='ano_semana', 
    y='casos_estimados', 
    hue='virus_nome',
    palette=cores_gripe,
    linewidth=2.2
)

semanas_unicas = sorted(df_gripe_br['ano_semana'].unique())
indices_ticks = np.linspace(0, len(semanas_unicas) - 1, 14, dtype=int)
plt.xticks([semanas_unicas[i] for i in indices_ticks], rotation=45)
plt.title('Evolução das Gripes no Brasil (Influenza A vs Influenza B)', fontsize=14, fontweight='bold')
plt.xlabel('Semana Epidemiológica')
plt.ylabel('Casos Estimados')
plt.legend(title='Agente')
plt.tight_layout()
plt.show()

### Sazonalidade Anual das Gripes (Picos por Semana Epidemiológica)

Sobreposição das curvas das semanas 1 a 53 para cada ano, permitindo observar claramente a sazonalidade (outono/inverno).

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(
    data=df_gripe_br,
    x='semana_epi',
    y='casos_estimados',
    hue='ano_epi',
    palette='tab10',
    marker='o',
    errorbar=None
)
plt.title('Sazonalidade da Gripe no Brasil (Casos Semanais por Ano)', fontsize=13, fontweight='bold')
plt.xlabel('Semana Epidemiológica (1 a 53)')
plt.ylabel('Casos Estimados')
plt.legend(title='Ano')
plt.tight_layout()
plt.show()

## 4. Comparativo Regional: Estados com Maior Volume de Casos de Gripe

Avaliamos quais estados concentram o maior número de casos de Influenza no ano epidemiológico mais recente.

In [ ]:
ano_recente = df_virus['ano_epi'].max()
df_estados_gripe = df_virus[
    (df_virus['uf_sigla'] != 'BR') & 
    (df_virus['ano_epi'] == ano_recente) & 
    (df_virus['virus'].isin(['FLU_A', 'FLU_B']))
]

total_por_uf = df_estados_gripe.groupby(['uf_sigla', 'virus_nome'])['casos_estimados'].sum().unstack().fillna(0)
total_por_uf['Total'] = total_por_uf.sum(axis=1)
top_ufs = total_por_uf.sort_values('Total', ascending=False).head(15)

plt.figure(figsize=(12, 6))
top_ufs[['Influenza A (Gripe A)', 'Influenza B (Gripe B)']].plot(
    kind='bar', 
    stacked=True, 
    color=['#d62728', '#1f77b4'],
    ax=plt.gca()
)
plt.title(f'Top 15 Estados com Maior Volume de Casos de Gripe em {ano_recente}', fontsize=13, fontweight='bold')
plt.xlabel('Estado (UF)')
plt.ylabel('Total de Casos Estimados')
plt.legend(title='Agente')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Efeito do Nowcasting: Casos Registrados vs Casos Estimados

O modelo estatístico de *nowcasting* da Fiocruz corrige o atraso de digitação nos hospitais. Nas semanas mais recentes, o número registrado ainda é baixo, mas o número estimado projeta o cenário real.

In [ ]:
ultimas_semanas = df_br[df_br['virus'] == 'FLU_A'].tail(25)

plt.figure(figsize=(14, 5))
plt.plot(ultimas_semanas['ano_semana'], ultimas_semanas['casos_estimados'], color='#d62728', label='Casos Estimados (Nowcasting Fiocruz)', marker='o', linewidth=2)
plt.plot(ultimas_semanas['ano_semana'], ultimas_semanas['casos_registrados'], color='gray', linestyle='--', label='Casos Registrados (Brutos)', marker='s')
plt.fill_between(ultimas_semanas['ano_semana'], ultimas_semanas['li'], ultimas_semanas['ls'], color='#d62728', alpha=0.15, label='Intervalo de Confiança (95%)')

plt.title('Nowcasting InfoGripe: Comparação entre Registrados e Estimados (Influenza A - BR)', fontsize=13, fontweight='bold')
plt.xlabel('Ano e Semana Epidemiológica')
plt.ylabel('Casos')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## 6. Conclusões e Próximos Passos

- O InfoGripe permite acompanhar não só a Gripe clássica (Influenza A e B), mas também seu comportamento relativo em relação a SARS-CoV-2 e VSR.
- O modelo de nowcasting é fundamental para gestores de saúde, pois evita a falsa sensação de queda na curva epidêmica causada apenas pela demora no envio das fichas hospitalares.
- Para estender a análise, utilize o arquivo `faixa_etaria_brasil_estados.csv` para estratificar por idade (impacto em crianças e idosos) e `alerta_semanal.csv` para mapas de alerta.